# Instant gratification

## An agent harness from first principles

In this notebook we build an AI agent that finds its way out of a maze.

Everything here runs on free models. The only setup is an OpenRouter account and an API key.

## Setup part 1: create your OpenRouter account

OpenRouter is a single service that gives you access to hundreds of models, including free ones. We use it throughout the course.

If you already have an OpenRouter account, sign in and go straight to part 2.

1. Open a new browser tab and go to **https://openrouter.ai**
2. Click the **Sign Up** button at the top right of the page.
3. Create your account with the Google button, the GitHub button or your email address. If you choose email, click the link in the verification email that OpenRouter sends you.
4. You do not need to add a credit card or buy credits for this notebook.

## Setup part 2: allow free models in your privacy settings

Free models will not run until you have done this step.

The providers of free models may train on your prompts or publish them. Nothing we send in this course is private.

1. Make sure you are signed in to OpenRouter, then go to **https://openrouter.ai/settings/privacy**
2. Find the setting called **Enable free endpoints that may train on inputs** and switch it **ON**.
3. Find the setting called **Enable free endpoints that may publish prompts** and switch it **ON**.
4. Check that both switches are on.

## Setup part 3: create your API key

If you already have an OpenRouter key saved from another course, use it and go straight to part 4.

1. Go to **https://openrouter.ai/settings/keys**
2. Click the **Create API Key** button.
3. In the **Name** box, type `fde-course`. Leave everything else as it is.
4. Click **Create**.
5. Your new key appears on the screen. It is a long piece of text that starts with `sk-or-v1-`
6. Copy the whole key using the copy button next to it. **OpenRouter only shows you the key once.** Keep this tab open until you have finished part 4. If you lose the key, delete it on this page and create another one.

Never paste your key into a notebook cell. The next step stores it in a `.env` file instead.

## Setup part 4: add the key to a `.env` file

1. In VS Code, right-click in the empty space below the list of files in the Explorer on the left, and choose **New File**.
2. Name the file exactly `.env` (not `env` or `.env.txt`). It goes in the `undercover` folder, next to `pyproject.toml`.
3. In the file, type this exactly, in capital letters, with underscores and with no spaces:

   `OPENROUTER_API_KEY=`

4. Paste your key straight after the equals sign (Ctrl+V on a PC, Cmd+V on a Mac), with no spaces or quotes. The line should look like `OPENROUTER_API_KEY=sk-or-v1-...` with the rest of your key in place of the dots.
5. Save the file (Ctrl+S on a PC, Cmd+S on a Mac).

Git ignores the `.env` file and never commits your key.

## Now for the code

Click **Select Kernel** at the top right of this notebook, choose **Python Environments**, and pick `.venv` in the `undercover` folder. If there is no `.venv`, run `uv sync` in a terminal in the `undercover` folder and try again.

Run each cell in turn by clicking into it and pressing **Shift + Enter**.

In [1]:
import json
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

### Check your key

This cell reads your key from the `.env` file.

If you see **No OpenRouter API key found**, check that the file is called exactly `.env`, that it is in the `undercover` folder, that the line starts `OPENROUTER_API_KEY=` and that you saved it (Setup part 4).

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key and openrouter_api_key.startswith("sk-or-v1-"):
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:9]}")
else:
    print("No OpenRouter API key found. Please check your .env file (Setup part 4).")

OpenRouter API key exists and begins sk-or-v1-


### Connect to OpenRouter

We use the OpenAI Python package and point it at OpenRouter by changing the base URL.

The model `openrouter/free` picks one of the available free models for each request. We print the name of the model that answered.

In [3]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

In [4]:
messages = [{"role": "user", "content": "Hi there!"}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by poolside/laguna-s-2.1:free


Hello! How can I help you today?

If you see a reply, everything is set up.

If you see an error that mentions **data policy** or **No endpoints found**, go back to Setup part 2 and check that both switches are on. If you see an error that mentions **401**, repeat Setup parts 3 and 4 to get a new key.

## Part 1 of the harness: the tool

A tool is a Python function that the model can ask us to run.

Our agent has one tool, `read_web_page`. It returns the text of a web page without the HTML. It only reads pages in the maze. For any other address, it tells the model which URLs it can read.

In [5]:
MAZE_URL = "https://maze.edwarddonner.com"

def read_web_page(url):
    """Read a web page in the maze and return the text on the page, without the HTML"""
    print(f"Reading web page: {url}")
    if not url.startswith(MAZE_URL):
        return f"Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with {MAZE_URL}"
    response = requests.get(url, timeout=20)
    page = BeautifulSoup(response.text, "html.parser")
    return page.body.get_text(separator="\n", strip=True)

In [6]:
print(read_web_page("https://maze.edwarddonner.com"))

Reading web page: https://maze.edwarddonner.com
The Maze
This page is for an AI that can read web pages. Your goal is to find the way out of a small maze and bring back the message that is waiting there. Every location in the maze is a web page that describes the room and lists its exits.
Step 1 is the entrance:
Location 1
https://maze.edwarddonner.com/1
Start with the maze now. Open the entrance, then follow the exits one page at a time. Some passages are dead ends, so go back and try another way when you reach one.
Keep the person you are working with updated as you go. After each location, say where you are and what you found there, and show a mini-map of the maze as you have seen it so far, drawn in text with the rooms you have visited and the exits you know about. When you reach the way out, give the message and a final map of your whole journey.
If you are a person reading this, give this address to ChatGPT or Claude and ask it to do what the page says.


Here is what happens with an address outside the maze.

In [7]:
print(read_web_page("https://edwarddonner.com"))

Reading web page: https://edwarddonner.com
Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with https://maze.edwarddonner.com


The model never sees our Python code. We describe the tool to it in JSON with the tool's name, what it does and the arguments it takes. The model reads this description and decides when to use the tool.

In [8]:
read_web_page_json = {
    "type": "function",
    "function": {
        "name": "read_web_page",
        "description": "Read a web page and return the text on the page",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "The full URL of the web page to request"},
            },
            "required": ["url"],
        },
    },
}

tools = [read_web_page_json]

When the model wants to use the tool, it replies with the tool's name and the arguments as JSON. Our code runs the function and sends back the result.

If the tool fails, we send the error message back to the model as the result.

In [9]:
def handle_tool_call(tool_call):
    """Run the tool that the model asked for and package the result as a message"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        result = read_web_page(**arguments)
    except Exception as error:
        result = f"The tool call failed: {error}"
    return {"role": "tool", "tool_call_id": tool_call.id, "content": result}

## Using the tool once

Now we call the model with the tool and ask it to describe the maze's front page. The model replies with a request to use the tool.

In [10]:
messages = [{"role": "user", "content": "Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now, without going into the maze."}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message

print(f"Answered by {response.model}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(message.tool_calls)

Answered by cohere/north-mini-code:free
Finish reason: tool_calls
[ChatCompletionMessageFunctionToolCall(id='read_web_page_dmgqzd4y613g', function=Function(arguments='{"url": "https://maze.edwarddonner.com"}', name='read_web_page'), type='function', index=0)]


The finish reason is `tool_calls`. The model has asked for `read_web_page` with the maze's address.

We add the model's request and the tool's result to the conversation and call the model again.

Now and then a small model asks for the tool again instead of answering. If the output is blank, run the two cells again.

In [11]:
messages.append(message)
for tool_call in message.tool_calls:
    messages.append(handle_tool_call(tool_call))

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Reading web page: https://maze.edwarddonner.com
Answered by poolside/laguna-s-2.1:free


<IPython.core.display.Markdown object>

To walk the maze, the model needs a tool call for every location it visits.

## Part 2 of the harness: the loop

First, a system prompt that asks the model to reply in Markdown.

In [12]:
SYSTEM_PROMPT = """
You are an agent that completes tasks by using your tools, one step at a time.
Your replies are displayed as Markdown, so format them in Markdown with headings, bold text and lists.
Put any map or diagram in a code block so that it lines up. Do not use emojis.
"""

The agent is the code from the last section inside a loop. The loop stops when the model replies with text and no tool call, or after 25 steps. If a free model sends back an empty reply, we ask again.

In [13]:
def run_agent(task, max_steps=25):
    """Call the model in a loop, running the tool whenever it asks, until it gives a final answer"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        print(f"Step {step} answered by {response.model}")

        text = (message.content or "").strip()
        if text:
            display(Markdown(text))

        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                messages.append(handle_tool_call(tool_call))
        elif text:
            return
        else:
            print("The reply was empty, so we ask again")

    print("The agent ran out of steps")

## Into the maze

I have built a small maze at **https://maze.edwarddonner.com**

Each location is a web page that describes a room and lists its exits. Some passages are dead ends. There is a message waiting at the way out.

We give the agent the maze's address and ask it to do what the page says. A run takes a minute or two. If you see an error, or the agent stops before it finds the way out, run the cell again.

In [14]:
run_agent("Have a look at https://maze.edwarddonner.com and do what it says.")

Step 1 answered by nvidia/nemotron-3-ultra-550b-a55b:free
Reading web page: https://maze.edwarddonner.com
Step 2 answered by minimax/minimax-m3.1-flash:free


I'll start the maze now. Let me open the entrance.

Reading web page: https://maze.edwarddonner.com/1
Step 3 answered by cohere/north-mini-code:free
Reading web page: https://maze.edwarddonner.com/14
Step 4 answered by nvidia/nemotron-3.5-lightning:free
Reading web page: https://maze.edwarddonner.com/62
Step 5 answered by minimax/minimax-m3.1-flash:free
Reading web page: https://maze.edwarddonner.com/39
Step 6 answered by minimax/minimax-m3.1-flash:free
Reading web page: https://maze.edwarddonner.com/71
Step 7 answered by nvidia/nemotron-3.5-super-vl:free


Location 71 is a dead end. I'll backtrack to location 62 and try the east exit to location 48.

Reading web page: https://maze.edwarddonner.com/48
Step 8 answered by poolside/laguna-s-2.1:free
Reading web page: https://maze.edwarddonner.com/83
Step 9 answered by poolside/laguna-s-2.1:free
Reading web page: https://maze.edwarddonner.com/25
Step 10 answered by nvidia/nemotron-3.5-lightning:free
Reading web page: https://maze.edwarddonner.com/35
Step 11 answered by poolside/laguna-s-2.1:free


I found the way out! Here is a summary of my journey through the maze.

## Maze Solution

**Starting at Location 1** (the entrance), I navigated north through the maze. The route to the exit was:

1 → 14 → 62 → 71 (dead end, backtracked to 62) → 48 → 83 → 25 → 35 (exit)

## Final Map

```
                [71] (Dead End)
                 |
                 | N
                 v
[1] --N/--14 --N/--62 --E/--83 --N/--25 --E/--35 (EXIT)
 ^                    |         ^          ^
 |                    |         | E        |
 E from 1             |         v          | S
 [27]                  | W   [56]            |
                       |                     |
 [39]<--W--[71]? No, 71 is E of 39 --E------83
   ^                    ^           ^          ^
   | E                  |           | E        |
   v                    | W         v          |
... (unexplored)         v
                      [48] --E/--25

```

Let me simplify with a cleaner, corrected text map:

```
                        [71]* DEAD END
                          |
                       W  |  E
                   *------[39]
                              |
                        S     |  S
                              |
      [27]*     E       [83] ====> [25] --E--> [35] EXIT
         \           /   |   \       |
          \       /     |    \      |
           \   /       |     \     |
          E [1] --N--> [14] --N--> [62] --E--> [48]
            |                     |
            |                     |
          (unexplored south/east exits, etc.)
```

Actually, here is a corrected, cleaner map showing only what I explored:

```
                        [71]* (DEAD END)
                          |
                       (W)|
                          |
                          |
(27)* <----E---- [1] --N---> [14] --N---> [62] --E---> [48] --E---> [83]
                               ^  S                              ^  W
                               |                                  |
                            (14)                                (48)
                                                        N
                                                        |
                                                        v
                                                     [25] --E---> [35] EXIT
                                                       |
                                                    (S back
                                                     to 83)
```

## The Message

> **"Welcome to the start of the course! Prepare for 3 intensive weeks ahead."**

I followed the northern path from the entrance, hit one dead end at location 71 (a broom closet), backtracked, continued east, and found the exit at location 35 after climbing a staircase into the daylight.

## What just happened

The model read each page, picked an exit, backed out of dead ends and carried on until it found the message.

Several different models probably took turns answering. Each one worked from the same conversation history.

The code around the model is called the agent harness. We return to harness engineering at the end of Week 1.

### Try it yourself

- Run the last cell again. Does the agent take the same route?
- Change `MODEL` to one named free model from https://openrouter.ai/models?max_price=0 and compare the number of steps it takes.

A free OpenRouter account allows 50 requests a day to free models. A run of the maze uses 10 to 15.

## Congratulations! And a very quick request.

You've got your agent harness working nicely locally. This is the start of a lot of Agent Harness work.

Much more LLM calling coming tomorrow!

**I should mention: I realize it's very early in the course, but if you'd be willing to rate the course in Udemy, I'd be super grateful. It's one of the main ways Udemy decides whether to show the course to others and it makes a huge difference. Thank you!**
